# GourmetAI — Classificazione di immagini di cibo

Progetto didattico di **Computer Vision** basato su **Transfer Learning** e **PyTorch**. Classifica automaticamente immagini di piatti in **14 categorie alimentari**, confrontando 6 architetture di reti neurali e selezionando la migliore in base alla validation accuracy.

---

## Obiettivi

- Addestrare una **CNN baseline** da zero come benchmark
- Confrontare **6 modelli di Transfer Learning** (VGG16, ResNet50, EfficientNet-B0/B4, ConvNeXt-Tiny, Swin-Tiny)
- Selezionare automaticamente il **modello migliore** (per val_acc)
- Valutare sul test set con confusion matrix e classification report
- Generare un **report HTML** con tutti i risultati
- **Fault tolerance**: checkpoint progressivi per riprendere il training dopo crash o timeout (es. Google Colab)

---

## Dataset

- **14 000 immagini** (train: 8 960 | val: 2 240 | test: 2 800)
- **14 classi**: Baked Potato, Crispy Chicken, Donut, Fries, Hot Dog, Sandwich, Taco, Taquito, Apple Pie, Cheesecake, Chicken Curry, Ice Cream, Omelette, Sushi
- **Link pubblico**: https://proai-datasets.s3.eu-west-3.amazonaws.com/dataset_food_classification.zip
- **Struttura**: `dataset/train/`, `dataset/val/`, `dataset/test/` con sottocartelle per classe
- **Setup Colab**: la cella successiva scarica e decomprime automaticamente il dataset se non presente

---

## Pipeline

1. **Setup**: import, seed, device (CPU/MPS/CUDA), trasformazioni (baseline + aug_strong)
2. **Baseline CNN**: addestramento da zero, valutazione su val/test
3. **Transfer Learning**: per ogni modello candidato:
   - **Head-only** (25 epoche): backbone congelato, solo classificatore
   - **Fine-tuning** (30 epoche): ultimi blocchi sbloccati, LR più basso
   - Checkpoint best model + checkpoint periodico ogni N epoch (fault tolerance)
4. **Confronto**: tabella val_acc/test_acc, scelta del migliore
5. **Valutazione finale**: confusion matrix, classification report, curve di training
6. **Report HTML**: generato automaticamente in `reports/`

---

## Checkpoint e fault tolerance

- **Best model**: `checkpoints/{modello}/best_head.pt`, `best_finetune.pt` (salvati da EarlyStopping)
- **Checkpoint periodico**: `resume_head.pt`, `resume_finetune.pt` (ogni `checkpoint_every` epoch, default 5)
- **Ripristino**: in caso di interruzione, rieseguire la cella del loop TL; il training riprende automaticamente dall'ultimo checkpoint salvato

---

## Modelli confrontati

| Modello | Anno | Tipo |
|---------|------|------|
| VGG16 | 2014 | CNN classica |
| ResNet50 | 2015 | Skip connections |
| EfficientNet-B0 | 2019 | Scaling uniforme |
| EfficientNet-B4 | 2019 | Upgrade di B0 |
| ConvNeXt-Tiny | 2022 | CNN modernizzata |
| Swin-Tiny | 2021 | Vision Transformer |

---

## Esecuzione

- **Google Colab**: caricare il notebook, eseguire tutte le celle; la cella di setup installa albumentations e scarica il dataset
- **Locale**: dataset in `dataset/`, `pip install torch torchvision albumentations matplotlib seaborn scikit-learn`
- **Config**: `cfg = Config()` per default; `cfg = Config(dataset_dir="...", checkpoint_every=3)` per personalizzare

---

## Compatibilità hardware

- **GPU NVIDIA (CUDA)**: pieno supporto
- **CPU**: supporto (lento)
- **Apple Silicon (MPS)**: supporto con workaround automatico per ConvNeXt e Swin (fine-tuning su CPU per bug PyTorch MPS)

In [ ]:
from IPython.display import Image, display
import base64

mermaid_code = """
flowchart TD
    subgraph setup [Setup]
        A[Dataset train/val/test]
        B[Config e seed]
        C[Trasformazioni baseline e aug]
    end
    subgraph train [Training]
        D[Baseline CNN benchmark]
        E[Lista modelli candidati]
        F[Transfer learning: solo head]
        G[Fine-tuning: ultimi blocchi]
    end
    subgraph eval [Valutazione]
        H[Riepilogo e scelta migliore]
        I[Curve di training]
        J[Test set: confusion matrix e report]
    end
    A --> B --> C
    C --> D
    C --> E
    E --> F --> G
    G --> H --> I --> J
"""

encoded = base64.urlsafe_b64encode(mermaid_code.encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}"))

### 0) Setup ambiente (Google Colab)

Questa cella è necessaria **solo su Google Colab**.
Installa le dipendenze `albumentations` e `gdown` (non presenti di default su Colab)
e scarica automaticamente:
- il **dataset** pubblico dal link ufficiale del corso
- la **documentazione** (README.md, config_exp01/02/03.md) dal tuo Google Drive, se configurato

**Dataset:**
- Se il dataset è già presente nella sessione, il download viene saltato.
- Su un ambiente locale con il dataset già nella cartella `dataset/`, questa cella non fa nulla.
- Il file zip viene eliminato dopo l'estrazione per liberare spazio.

**Documentazione:**
- Sono inclusi un file README e 3 file contenenti tre diversi set di esperimenti con diversi parametri e confronti tra modelli. Il terzo esperimento contiene tutti e 6 i modelli confrontati.

**ATTENZIONE:** il terzo esperimento ha richiesto circa 15 ore di training su Apple MacBook M3 PRO con supporto MPS

In [ ]:
import os, urllib.request, zipfile

# Installa albumentations (non inclusa di default su Google Colab)
try:
    import albumentations
except ImportError:
    print("Installazione albumentations...")
    os.system("pip install albumentations -q")
    print("albumentations installata.")

# Installa gdown (per scaricare da Google Drive)
try:
    import gdown
except ImportError:
    os.system("pip install gdown -q")
    import gdown

# Scarica il dataset solo se non è già presente
DATASET_URL = "https://proai-datasets.s3.eu-west-3.amazonaws.com/dataset_food_classification.zip"
DATASET_DIR = "dataset"

if not os.path.exists(DATASET_DIR):
    print("Scaricamento dataset in corso (può richiedere qualche minuto)...")
    urllib.request.urlretrieve(DATASET_URL, "dataset.zip")
    print("Estrazione archivio...")
    with zipfile.ZipFile("dataset.zip", "r") as z:
        z.extractall(".")
    os.remove("dataset.zip")
    print(f"Dataset pronto in '{DATASET_DIR}/'")
else:
    print(f"Dataset già presente in '{DATASET_DIR}/', download saltato.")

# Documentazione: README.md e config esperimenti (da Google Drive)

DOCS_DRIVE_ID = "1Ys59d_ikcIT9pPYqu3Ba2-9u3fgH4lQv"
DOCS_FILES = ["README.md", "config_exp01.md", "config_exp02.md", "config_exp03.md"]

if DOCS_DRIVE_ID and not all(os.path.exists(f) for f in DOCS_FILES):
    print("Scaricamento documentazione da Drive...")
    gdown.download(id=DOCS_DRIVE_ID, output="gourmetai_docs.zip", quiet=False)
    with zipfile.ZipFile("gourmetai_docs.zip", "r") as z:
        z.extractall(".")
    os.remove("gourmetai_docs.zip")
    print("Documentazione pronta.")
elif DOCS_DRIVE_ID:
    print("Documentazione già presente.")

1. **Setup**: caricamento della struttura dataset, configurazione (Config), seed per riproducibilità, definizione delle trasformazioni (baseline per val/test, augmentation per il training).
2. **Training**: addestramento della CNN baseline (senza Transfer Learning o TL); per ogni modello candidato, fase di transfer learning (solo testa) seguita da fine-tuning (backbone parzialmente sbloccato), con early stopping e salvataggio dei checkpoint.
3. **Valutazione**: confronto delle metriche (val_acc, test_acc), scelta del modello migliore in base alla validation, visualizzazione delle curve e valutazione finale sul test set (matrice di confusione, precision, recall, F1).

## Obiettivi

- **Riproducibilità**: seed fissato e pipeline deterministico dove possibile.
- **Confronto multi-modello**: confrontare in modo equo più architetture (VGG16, ResNet18, ResNet50, EfficientNet-B0) sullo stesso dataset e con le stesse impostazioni.
- **Scelta del modello**: selezionare il modello con la **migliore validation accuracy** (senza usare il test per la scelta), poi valutare solo il migliore sul test set.
- **Interpretabilità**: curve di training (loss e accuracy), matrice di confusione e classification report per analizzare errori e precisione per classe.

### 1) Import e dipendenze

La cella successiva importa tutte le librerie necessarie al progetto: PyTorch e torchvision per i modelli e i dati, Albumentations per le trasformazioni immagine, matplotlib e sklearn per visualizzazione e metriche.

> **Nota tecnica — `PYTORCH_ENABLE_MPS_FALLBACK`**: prima degli import viene impostata
> la variabile d'ambiente `PYTORCH_ENABLE_MPS_FALLBACK=1`. Su Apple Silicon (chip M1/M2/M3)
> questa istruzione abilita il fallback automatico CPU per le operazioni non ancora supportate
> dal backend MPS di PyTorch (ad esempio `AdaptiveAvgPool2d` con dimensioni non divisibili).
> Su altri sistemi (CUDA, CPU) la variabile viene ignorata.

In [ ]:
"""
GourmetAI - Classificazione di immagini di cibo.

Pipeline: dataset (train/val/test) -> baseline CNN -> augmentation -> transfer learning
-> fine-tuning -> test finale.

Componenti principali:
- Albumentations + wrapper Transforms per trasformazioni immagine
- ImageFolder per caricare dataset da cartelle
- train_epoch / test_epoch / train() per addestramento e valutazione
- EarlyStopping con salvataggio del miglior modello
- Matrice di confusione e classification report
"""

from __future__ import annotations

# Standard library: percorsi, casualità, dataclass, tipi, I/O, encoding, data e ora
import os
import random
import shutil
import base64
import io
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional

# NumPy e PyTorch: tensori, reti neurali, ottimizzazione
import numpy as np
import pandas as pd
# Deve essere impostata PRIMA di importare torch.
# Fallback MPS → CPU per operazioni non supportate su Apple Silicon (AdaptiveAvgPool, ConvNeXt backward, ecc.)
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Albumentations: trasformazioni e conversione in tensore
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Visualizzazione e metriche
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

### 2) Riproducibilità e device

Si definiscono **seed_everything** (per fissare il seed di Python, NumPy e PyTorch e ottenere risultati riproducibili) e **get_device** (per selezionare automaticamente il dispositivo: CUDA, MPS o CPU). Queste variabili/funzioni vengono usate nelle celle successive.

In [ ]:
# -----------------------------
# 0) Riproducibilità + Device
# -----------------------------

def seed_everything(seed: int = 42) -> None:
    """
    Fissa il seed per Python, NumPy e PyTorch (CPU e GPU) per ottenere
    risultati riproducibili tra diverse esecuzioni.
    Chiamare all'inizio del training o dello script.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    """
    Restituisce il dispositivo disponibile migliore: CUDA (GPU NVIDIA),
    MPS (GPU Apple) o CPU. L'ordine di preferenza evita di usare la CPU
    quando è disponibile una GPU.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


### 3) Wrapper Albumentations per ImageFolder

La classe **Transforms** permette di usare trasformazioni Albumentations (che lavorano su array NumPy) insieme a **torchvision.datasets.ImageFolder** (che passa immagini PIL). Il wrapper converte l’immagine in array, applica la pipeline Albumentations e restituisce il tensore.

In [ ]:
# -----------------------------
# 2) Albumentations wrapper (come notebook)
# -----------------------------

class Transforms:
    """
    Wrapper per usare trasformazioni Albumentations con torchvision ImageFolder.
    ImageFolder passa immagini PIL; Albumentations richiede array NumPy.
    __call__ converte PIL -> NumPy, applica la pipeline e restituisce il tensore.
    """
    def __init__(self, transforms: A.Compose):
        # Pipeline Albumentations (resize, augment, normalize, ToTensorV2)
        self.transforms = transforms

    def __call__(self, img, *args, **kwargs):
        # Converti PIL in array NumPy; Albumentations restituisce dict con chiave "image"
        return self.transforms(image=np.array(img))["image"]

### 4) Trasformazioni: baseline e augmentation

La funzione **build_transforms** costruisce le pipeline di trasformazione in tre modalita':
- **baseline**: resize + normalizzazione ImageNet (usata per val e test).
- **aug_light**: baseline + flip orizzontale, rotazione 90 gradi, shift/scale/rotate, random crop e color jitter leggero (per il training).
- **aug_strong**: aug_light + prospettiva, blur, rumore Gaussiano e coarse dropout (augmentation piu' aggressiva).

**Normalizzazione**: vengono definite le costanti `IMAGENET_MEAN` e `IMAGENET_STD` con i valori ufficiali di ImageNet `(0.485, 0.456, 0.406)` e `(0.229, 0.224, 0.225)`. Questi valori sono **obbligatori** per tutti i backbone preaddestrati (VGG, ResNet, EfficientNet, ConvNeXt, Swin): usare valori diversi degrada le feature estratte dal backbone e penalizza la precisione del transfer learning.

In [ ]:
# -----------------------------
# 3) Transforms: baseline + augmentation
# -----------------------------

# Statistiche di normalizzazione ImageNet: richieste da tutti i backbone preaddestrati
# (VGG, ResNet, EfficientNet, ConvNeXt, Swin). Usare valori diversi degrada le feature.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def build_transforms(
    image_size: int = 224,
    mode: str = "baseline",
) -> A.Compose:
    """
    Costruisce una pipeline di trasformazioni Albumentations.

    Parametri
    ----------
    image_size : lato (in pixel) dell'immagine in uscita (quadrata).
    mode       : modalita' di trasformazione:
        - "baseline"  : solo resize e normalizzazione ImageNet (media/std corretti per TL).
        - "aug_light" : baseline + flip orizzontale, rotazione 90 gradi, shift/scale/rotate,
          random resized crop e color jitter leggero (per training).
        - "aug_strong": aug_light + prospettiva, blur, rumore Gaussiano e coarse
          dropout (data augmentation piu' aggressiva).

    Normalizzazione: IMAGENET_MEAN=(0.485,0.456,0.406) / IMAGENET_STD=(0.229,0.224,0.225).
    Questi valori sono obbligatori per i backbone preaddestrati su ImageNet.

    Restituisce
    -----------
    Oggetto A.Compose da passare al wrapper Transforms o ai dataset.
    """
    # Trasformazioni comuni a tutte le modalita': resize e normalizzazione ImageNet
    base = [
        A.Resize(image_size, image_size),
        A.Normalize(IMAGENET_MEAN, IMAGENET_STD),  # valori corretti per TL preaddestrato
    ]

    if mode == "baseline":
        return A.Compose(base + [ToTensorV2()])

    if mode == "aug_light":
        aug = [
            A.HorizontalFlip(p=0.5),
            A.RandomRotate90(p=0.3),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=15, p=0.5),
            A.RandomResizedCrop(size=(image_size, image_size), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.5),
            A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05, p=0.4),
        ]
        return A.Compose(base + aug + [ToTensorV2()])

    if mode == "aug_strong":
        aug = [
            A.HorizontalFlip(p=0.5),
            A.RandomRotate90(p=0.3),
            A.ShiftScaleRotate(shift_limit=0.07, scale_limit=0.15, rotate_limit=20, p=0.6),
            A.RandomResizedCrop(size=(image_size, image_size), scale=(0.75, 1.0), ratio=(0.85, 1.15), p=0.6),
            A.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.20, hue=0.08, p=0.5),
            A.Perspective(scale=(0.02, 0.05), p=0.2),
            A.GaussianBlur(blur_limit=(3, 5), p=0.15),
            A.GaussNoise(var_limit=(5.0, 25.0), p=0.15),
            A.CoarseDropout(max_holes=8, max_height=24, max_width=24, fill_value=0, p=0.2),
        ]
        return A.Compose(base + aug + [ToTensorV2()])

    raise ValueError(f"mode non valido: {mode}")


### 5) Modello CNN baseline

Si definisce la rete **BaselineCNN**: una CNN semplice (conv + pool + fully connected) usata come benchmark senza transfer learning. Il numero di classi è passato al costruttore. Viene usata `AdaptiveAvgPool2d` per essere robusti rispetto alla dimensione dell’input.

In [ ]:
# -----------------------------
# 4) Baseline model (come notebook: CNN semplice)
# -----------------------------

class BaselineCNN(nn.Module):
    """
    CNN baseline semplice: 3 blocchi convoluzionali (conv + ReLU + pool) seguiti
    da pooling adattivo e due layer fully connected con dropout.
    Usata come benchmark senza transfer learning; AdaptiveAvgPool2d rende la rete
    indipendente dalla dimensione spaziale in input (basta che sia >= qualche pixel).
    """
    def __init__(self, num_classes: int):
        super().__init__()
        # Blocco 1: 3 canali -> 32 feature maps, kernel 5x5
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
        )
        # Pooling adattivo: output sempre 4x4 indipendentemente dalla size in input
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        # Classificatore: 128*4*4 -> 256 -> num_classes, con dropout 0.3
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


### 5a) Modelli per Transfer Learning

Questa cella definisce la funzione `build_transfer_model` che supporta sei architetture:
**VGG16, ResNet50, EfficientNet-B0, EfficientNet-B4, ConvNeXt-Tiny e Swin-Tiny**.
Ogni modello viene caricato con pesi preaddestrati su ImageNet tramite `torchvision.models`;
si sostituisce direttamente il layer di classificazione finale (senza wrapper custom)
con una testa adatta al numero di classi del dataset.
La funzione `unfreeze_last_blocks` sblocca gli ultimi blocchi del backbone per la fase di fine-tuning.

In [ ]:
# -----------------------------
# 5) Transfer Learning models
# -----------------------------
# VGGClassifier rimosso: ora si usa torchvision.models.vgg16 direttamente in build_transfer_model


def build_transfer_model(
    name: str,
    num_classes: int,
    device: torch.device,
    freeze_backbone: bool = True,
) -> nn.Module:
    """
    Costruisce un modello per transfer learning: backbone preaddestrato su ImageNet
    con l'ultimo layer sostituito da una testa adatta al numero di classi.

    Parametri
    ----------
    name            : architettura da usare.
                      Valori validi: 'vgg16' | 'resnet50' | 'efficientnet_b0' |
                      'efficientnet_b4' | 'convnext_tiny' | 'swin_tiny'
    num_classes     : numero di classi del dataset
    device          : device su cui caricare il modello (cpu / cuda / mps)
    freeze_backbone : se True, congela tutti i parametri tranne la testa

    Restituisce
    -----------
    nn.Module : modello pronto per il training della sola testa
    """
    name = name.lower()

    # --- VGG16: backbone classico (2014), sostituisce model.classifier[6] ---
    # Si usa il modello torchvision direttamente (come tutti gli altri),
    # evitando wrapper custom non necessari e problemi MPS con AdaptiveAvgPool2d.
    if name == "vgg16":
        from torchvision.models import vgg16, VGG16_Weights
        model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).to(device)
        if freeze_backbone:
            for p in model.parameters():
                p.requires_grad = False
        # model.classifier e' un Sequential con 6 layer; l'ultimo (indice 6) e' Linear(4096, 1000)
        in_features = model.classifier[6].in_features  # 4096
        model.classifier[6] = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(in_features, num_classes),
        ).to(device)
        for p in model.classifier[6].parameters():
            p.requires_grad = True
        return model

    # --- ResNet50: skip connections (2015), sostituisce il layer fc finale ---
    if name == "resnet50":
        from torchvision.models import resnet50, ResNet50_Weights
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(device)
        if freeze_backbone:
            for p in model.parameters():
                p.requires_grad = False
        # Sostituisce il layer fc con Dropout + Linear
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes),
        ).to(device)
        for p in model.fc.parameters():
            p.requires_grad = True
        return model

    # --- EfficientNet-B0: scaling uniforme compatto (2019), testa in model.classifier ---
    if name == "efficientnet_b0":
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).to(device)
        if freeze_backbone:
            for p in model.parameters():
                p.requires_grad = False
        # Il classificatore di EfficientNet e' una lista: [Dropout, Linear]
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes),
        ).to(device)
        for p in model.classifier.parameters():
            p.requires_grad = True
        return model

    # --- EfficientNet-B4: versione piu' grande di B0, stessa struttura (2019) ---
    if name == "efficientnet_b4":
        from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
        model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1).to(device)
        if freeze_backbone:
            for p in model.parameters():
                p.requires_grad = False
        # Stessa struttura del classificatore di B0
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes),
        ).to(device)
        for p in model.classifier.parameters():
            p.requires_grad = True
        return model

    # --- ConvNeXt-Tiny: CNN modernizzata ispirata ai Transformer (2022) ---
    # La testa si trova in model.classifier[2] (LayerNorm -> Flatten -> Linear)
    if name == "convnext_tiny":
        from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
        model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1).to(device)
        if freeze_backbone:
            for p in model.parameters():
                p.requires_grad = False
        # Sostituisce solo il layer Linear finale (indice 2 nel classifier)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes).to(device)
        for p in model.classifier[2].parameters():
            p.requires_grad = True
        return model

    # --- Swin-Tiny: Swin Transformer con attention a finestre locali (2021) ---
    # La testa di classificazione si chiama model.head
    if name == "swin_tiny":
        from torchvision.models import swin_t, Swin_T_Weights
        model = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1).to(device)
        if freeze_backbone:
            for p in model.parameters():
                p.requires_grad = False
        # Sostituisce il layer head con uno adatto al numero di classi
        in_features = model.head.in_features
        model.head = nn.Linear(in_features, num_classes).to(device)
        for p in model.head.parameters():
            p.requires_grad = True
        return model

    raise ValueError(f"Modello non supportato: {name}. Scegli tra: vgg16, resnet50, efficientnet_b0, efficientnet_b4, convnext_tiny, swin_tiny")


def unfreeze_last_blocks(model: nn.Module, name: str, num_blocks: int = 1) -> None:
    """
    Sblocca gli ultimi blocchi del backbone per il fine-tuning (requires_grad=True).
    Modifica il modello in-place. Chiamare dopo il training della sola testa,
    prima del fine-tuning.

    Parametri
    ----------
    model      : modello gia' costruito con build_transfer_model
    name       : nome dell'architettura (stessa stringa usata in build_transfer_model)
    num_blocks : numero di blocchi/stage finali da sbloccare (default=1)
    """
    name = name.lower()

    if name == "vgg16":
        # VGG: sblocca gli ultimi num_blocks*5 layer del blocco features (euristica)
        features = model.features
        to_unfreeze = list(features.children())[-(num_blocks * 5):]
        for layer in to_unfreeze:
            for p in layer.parameters():
                p.requires_grad = True

    elif name == "resnet50":
        # ResNet50: layer4 e' l'ultimo blocco residuale principale
        if hasattr(model, "layer4"):
            for p in model.layer4.parameters():
                p.requires_grad = True

    elif name in ("efficientnet_b0", "efficientnet_b4"):
        # EfficientNet B0/B4: backbone in model.features, sblocca gli ultimi num_blocks stage
        if hasattr(model, "features"):
            children = list(model.features.children())
            for layer in children[-num_blocks:]:
                for p in layer.parameters():
                    p.requires_grad = True

    elif name == "convnext_tiny":
        # ConvNeXt: backbone in model.features, sblocca gli ultimi num_blocks stage
        if hasattr(model, "features"):
            children = list(model.features.children())
            for layer in children[-num_blocks:]:
                for p in layer.parameters():
                    p.requires_grad = True

    elif name == "swin_tiny":
        # Swin Transformer: backbone in model.features, sblocca gli ultimi num_blocks stage
        if hasattr(model, "features"):
            children = list(model.features.children())
            for layer in children[-num_blocks:]:
                for p in layer.parameters():
                    p.requires_grad = True

    else:
        # Modello sconosciuto: nessuna azione
        return


### 6) Early stopping (fault-tolerant)

La classe **EarlyStopping** interrompe l'addestramento quando la loss di validazione non migliora per un numero di epoch pari a `patience`. Salva automaticamente i pesi del modello (best model) nel percorso indicato ogni volta che si raggiunge un nuovo minimo di validation loss.

**Novità v1 — serializzazione dello stato:**

Sono stati aggiunti i metodi `state_dict()` e `load_state_dict()` per serializzare e ripristinare lo stato interno (contatore, minimo corrente, flag `early_stop`). Questi metodi vengono utilizzati dal checkpoint periodico della funzione `train()` per garantire che, in caso di ripristino, l'early stopping riprenda esattamente dallo stesso punto e non azzeri il contatore.

In [ ]:
# -----------------------------
# 6) EarlyStopping (come notebook)
# -----------------------------

class EarlyStopping:
    """
    Interrompe il training quando la loss di validazione non migliora per
    `patience` epoch consecutivi. Salva su disco il checkpoint del modello
    ogni volta che si raggiunge un nuovo minimo di validation loss.
    """
    def __init__(self, save_path: str | Path, patience: int = 5, min_delta: float = 0.0):
        self.save_path = str(save_path)
        self.patience = patience
        self.min_delta = min_delta
        self.min_val_loss: Optional[float] = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, validation_loss: float, model: nn.Module) -> None:
        # Prima chiamata: salva sempre e memorizza la loss come minimo
        if self.min_val_loss is None:
            self.min_val_loss = validation_loss
            self.save_checkpoint(model)
            return

        # Miglioramento significativo (loss diminuita di almeno min_delta)
        if (self.min_val_loss - validation_loss) > self.min_delta:
            self.min_val_loss = validation_loss
            self.save_checkpoint(model)
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model: nn.Module) -> None:
        """Salva lo state_dict del modello al percorso indicato."""
        torch.save(model.state_dict(), self.save_path)

    def state_dict(self) -> dict:
        """
        Serializza lo stato interno di EarlyStopping.
        Permette di salvare il contatore e il minimo corrente nel checkpoint
        periodico, così il ripristino riprende esattamente dallo stesso punto.
        """
        return {
            "min_val_loss": self.min_val_loss,
            "counter": self.counter,
            "early_stop": self.early_stop,
        }

    def load_state_dict(self, state: dict) -> None:
        """
        Ripristina lo stato interno di EarlyStopping da un dizionario
        precedentemente salvato con state_dict().
        """
        self.min_val_loss = state["min_val_loss"]
        self.counter = state["counter"]
        self.early_stop = state["early_stop"]

### 7) Loop di training e valutazione (fault-tolerant)

Si definiscono **train_epoch** (un'epoch su training set), **test_epoch** (valutazione su un DataLoader con loss e accuracy) e **train** (ciclo completo con early stopping, scheduler opzionale e checkpoint periodico).

**Novità v1 — progressive checkpointing:**

La funzione `train()` accetta ora due parametri aggiuntivi:

- `resume_ckpt_path`: percorso di un checkpoint periodico da cui riprendere il training. Se il file esiste, la funzione carica automaticamente i pesi del modello, lo stato dell'ottimizzatore, lo stato dello scheduler, lo stato dell'early stopping e la history parziale, riprendendo dall'epoch successiva a quella salvata.
- `checkpoint_every`: ogni quante epoch salvare un checkpoint periodico (default `cfg.checkpoint_every = 5`). Se impostato a 0, il meccanismo è disabilitato.

**Struttura del checkpoint periodico** (file `resume_checkpoint.pt` nella stessa cartella del best model):
```
{
  "epoch": <int>,
  "model_state": <state_dict>,
  "optimizer_state": <state_dict>,
  "scheduler_state": <state_dict | None>,
  "early_stopping_state": {"min_val_loss", "counter", "early_stop"},
  "history": {"train_loss", "val_loss", "val_acc"}
}
```

> Nota: il best model (salvato da EarlyStopping) e il checkpoint periodico sono **due file separati**. Il best model (`best_head.pt`, `best_finetune.pt`) contiene solo i pesi del miglior modello; il checkpoint periodico contiene tutto il necessario per riprendere il training.

La storia completa (train_loss, val_loss, val_acc) viene restituita per i grafici.


In [ ]:
# -----------------------------
# 7) Train/Test loop
# -----------------------------

def train_epoch(
    model: nn.Module,
    train_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    """
    Esegue una singola epoch di training: forward, loss, backward, step.
    Restituisce la loss media sul training set (per batch la loss e' pesata
    con il numero di campioni nel batch).
    """
    model.train()
    running_loss = 0.0
    processed = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        # Gradient clipping: limita la norma dei gradienti per evitare aggiornamenti bruschi
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        processed += labels.size(0)

    return running_loss / max(processed, 1)


@torch.no_grad()
def test_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, float]:
    """
    Valuta il modello su un DataLoader (validation o test): nessun gradiente,
    accumulo di loss e conteggio predizioni corrette.
    Restituisce (loss_media, accuracy).
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    epochs: int,
    early_stopping: EarlyStopping,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,
    scheduler_on: str = "val_loss",          # "val_loss" | "epoch"
    resume_ckpt_path: Optional[str] = None,  # percorso checkpoint periodico da riprendere
    checkpoint_every: int = 0,               # salva ckpt periodico ogni N epoche (0 = disabilitato)
) -> Dict[str, List[float]]:
    """
    Loop di training completo con supporto fault-tolerance (progressive checkpointing).

    Parametri aggiuntivi rispetto a v0:
      resume_ckpt_path : se valorizzato, riprende il training dal checkpoint periodico
                         salvato a quel percorso (carica pesi modello, stato ottimizzatore,
                         stato scheduler, stato early-stopping e history parziale).
      checkpoint_every : ogni quante epoch salvare un checkpoint periodico (oltre al best-model
                         gestito da EarlyStopping). Se 0 il checkpoint periodico e' disabilitato.

    Il checkpoint periodico include:
      - state_dict del modello
      - state_dict dell'ottimizzatore
      - state_dict dello scheduler (se presente)
      - state_dict di EarlyStopping
      - history parziale (train_loss, val_loss, val_acc)
      - epoch corrente

    Restituisce la history completa con liste train_loss, val_loss, val_acc.
    """
    history: Dict[str, List[float]] = {"train_loss": [], "val_loss": [], "val_acc": []}
    start_epoch = 1

    # -----------------------------------------------------------------
    # RIPRISTINO: carica checkpoint periodico se indicato
    # -----------------------------------------------------------------
    if resume_ckpt_path and Path(resume_ckpt_path).exists():
        print(f"[RESUME] Carico checkpoint periodico da: {resume_ckpt_path}")
        ckpt = torch.load(resume_ckpt_path, map_location=device, weights_only=False)

        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        if scheduler is not None and "scheduler_state" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        if "early_stopping_state" in ckpt:
            early_stopping.load_state_dict(ckpt["early_stopping_state"])

        # Ripristina history parziale
        history = ckpt.get("history", history)
        start_epoch = ckpt.get("epoch", 0) + 1
        print(f"[RESUME] Riprendendo dall'epoch {start_epoch}")
    else:
        if resume_ckpt_path:
            print(f"[RESUME] Checkpoint non trovato ({resume_ckpt_path}), training da zero.")

    # -----------------------------------------------------------------
    # TRAINING LOOP
    # -----------------------------------------------------------------
    for epoch in range(start_epoch, epochs + 1):
        tr_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = test_epoch(model, val_loader, criterion, device)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if scheduler is not None:
            if scheduler_on == "val_loss" and hasattr(scheduler, "step"):
                if scheduler.__class__.__name__.lower().find("reducelronplateau") >= 0:
                    scheduler.step(val_loss)
                else:
                    scheduler.step()
            elif scheduler_on == "epoch":
                scheduler.step()

        early_stopping(val_loss, model)
        print(f"[Epoch {epoch:03d}] train_loss={tr_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        # -----------------------------------------------------------------
        # CHECKPOINT PERIODICO (fault tolerance)
        # -----------------------------------------------------------------
        if checkpoint_every > 0 and (epoch % checkpoint_every == 0):
            # Ricava il percorso del checkpoint periodico dallo stesso path di best model
            ckpt_dir = Path(early_stopping.save_path).parent
            periodic_path = ckpt_dir / "resume_checkpoint.pt"
            periodic_data = {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
                "early_stopping_state": early_stopping.state_dict(),
                "history": history,
            }
            torch.save(periodic_data, periodic_path)
            print(f"[CKPT] Checkpoint periodico salvato (epoch {epoch}): {periodic_path}")

        if early_stopping.early_stop:
            print("Early stopping triggered.")
            break

    return history

### 8) Confusion matrix, report e curve di training

Questa cella definisce: **predict_all** (raccoglie predizioni su un DataLoader), **plot_confusion_matrix** (disegna la matrice di confusione), **plot_training_curves** (disegna train/val loss e val accuracy per epoch, per diagnosticare overfitting e underfitting).

Codice delle funzioni descritte sopra (plot_training_curves, predict_all, plot_confusion_matrix).

In [ ]:
# -----------------------------
# 8) Confusion matrix + report + curve di training
# -----------------------------

def plot_training_curves(history: Dict[str, List[float]], title: str = "") -> None:
    """
    Disegna due grafici: (1) train loss e validation loss per epoch,
    (2) validation accuracy per epoch. Serve a diagnosticare overfitting
    (val loss che sale mentre train loss scende) e underfitting (curve alte).
    """
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    # Sottografico 1: train_loss e val_loss
    ax1.plot(epochs, history["train_loss"], label="Train loss", color="C0")
    ax1.plot(epochs, history["val_loss"], label="Val loss", color="C1")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.set_title(f"Loss - {title}" if title else "Loss")
    # Sottografico 2: val_acc
    ax2.plot(epochs, history["val_acc"], label="Val accuracy", color="C2")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.set_title(f"Validation accuracy - {title}" if title else "Validation accuracy")
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def predict_all(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    """
    Raccoglie le etichette vere (y_true) e le predizioni (y_pred) per tutti
    i batch del DataLoader. Il modello viene messo in eval() e non si
    calcolano gradienti. Restituisce due array NumPy concatenati.
    """
    model.eval()
    y_true, y_pred = [], []
    for inputs, labels in loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        y_pred.append(preds)
        y_true.append(labels.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: List[str],
    normalize: bool = False,
) -> None:
    """
    Disegna la matrice di confusione con annotazioni numeriche in ogni cella.

    Parametri
    ----------
    y_true       : etichette reali
    y_pred       : etichette predette
    class_names  : lista dei nomi delle classi (ordine uguale a quello dell'ImageFolder)
    normalize    : se True normalizza ogni riga per la somma (valori 0-1);
                   se False mostra i conteggi assoluti (default)

    Stile: colormap Blues, testo bianco su celle scure e nero su celle chiare,
    barra colori laterale, etichette assi ruotate.
    """
    # Calcola la matrice di confusione grezza
    cm = confusion_matrix(y_true, y_pred)

    # Normalizza per riga (recall per classe) se richiesto
    if normalize:
        cm_display = cm.astype(np.float32) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        fmt = ".2f"
    else:
        cm_display = cm
        fmt = "d"

    n = len(class_names)
    fig, ax = plt.subplots(figsize=(12, 10))

    # Disegna la heatmap con colormap blu
    im = ax.imshow(cm_display, cmap="Blues")
    plt.colorbar(im, ax=ax)

    # Soglia per scegliere il colore del testo (bianco su sfondo scuro, nero su sfondo chiaro)
    thresh = cm_display.max() / 2.0

    # Scrive il valore numerico al centro di ogni cella
    for i in range(n):
        for j in range(n):
            val = cm_display[i, j]
            label = format(val, fmt)
            ax.text(
                j, i, label,
                ha="center", va="center", fontsize=8,
                color="white" if val > thresh else "black",
            )

    # Etichette degli assi
    ax.set_xticks(range(n))
    ax.set_xticklabels(class_names, rotation=90, fontsize=10)
    ax.set_yticks(range(n))
    ax.set_yticklabels(class_names, fontsize=10)
    ax.set_xlabel("Predicted label", fontsize=12)
    ax.set_ylabel("True label", fontsize=12)
    ax.set_title("Confusion Matrix" + (" (normalized)" if normalize else ""), fontsize=14)

    plt.tight_layout()
    plt.show()

### 9) Configurazione e inizializzazione

Si definisce la dataclass **Config** con tutti i parametri (percorsi dataset/checkpoint, dimensioni immagine, batch size, learning rate, epoch, patience). Poi si inizializzano `cfg`, si imposta il seed, si ottiene il device e si verifica la struttura delle cartelle train/val/test (e eventuali incoerenze tra classi).

In [ ]:
# -----------------------------
# 9) Main orchestration (baseline -> aug -> TL -> fine-tune -> test)
# -----------------------------

@dataclass
class Config:
    """
    Configurazione centralizzata per il pipeline GourmetAI.
    Tutti i parametri hanno un default; modificare gli attributi o istanziare
    con valori diversi per cambiare percorsi, dimensioni, iperparametri di training.
    """
    dataset_dir: str = "dataset"       # cartella con sottocartelle train/val/test per classe
    checkpoint_dir: str = "checkpoints" # cartella base per checkpoint (sottocartelle per modello)
    image_size: int = 224
    batch_size: int = 64
    num_workers: int = 0
    seed: int = 42

    epochs_head: int = 25
    epochs_finetune: int = 30   # Exp03: piu' margine per convergenza

    lr_head: float = 3e-4
    lr_finetune: float = 5e-5

    weight_decay: float = 1e-3
    patience: int = 5   # Exp03: evita early stop troppo precoce con aug_strong
    checkpoint_every: int = 5       # salva checkpoint periodico ogni N epoche


def run_pipeline(cfg: Config) -> Tuple[torch.device, Path]:
    """
    Esegue seed, device e verifica struttura dataset (train/val/test) e coerenza
    delle classi tra split. Restituisce (device, dataset_dir) per uso nel notebook;
    il resto del flusso (dataset, training, confronto modelli) e' nelle celle successive.
    """
    seed_everything(cfg.seed)
    device = get_device()
    print("Device:", device)

    # Verifica che esistano le cartelle train, val, test
    dataset_dir = Path(cfg.dataset_dir)
    required = [dataset_dir / "train", dataset_dir / "val", dataset_dir / "test"]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Dataset non trovato o struttura non valida. Atteso: dataset/train, dataset/val, dataset/test. "
            f"Mancano: {missing}"
        )

    # Verifica coerenza delle classi (sottocartelle) tra train, val e test
    train_classes = {p.name for p in (dataset_dir / "train").iterdir() if p.is_dir()}
    val_classes = {p.name for p in (dataset_dir / "val").iterdir() if p.is_dir()}
    test_classes = {p.name for p in (dataset_dir / "test").iterdir() if p.is_dir()}

    missing_in_val = sorted(train_classes - val_classes)
    missing_in_test = sorted(train_classes - test_classes)
    extra_in_val = sorted(val_classes - train_classes)
    extra_in_test = sorted(test_classes - train_classes)

    if missing_in_val or missing_in_test or extra_in_val or extra_in_test:
        print("[WARN] Incoerenza cartelle classe tra train/val/test:")
        if missing_in_val:
            print("  - Missing in VAL:", missing_in_val)
        if missing_in_test:
            print("  - Missing in TEST:", missing_in_test)
        if extra_in_val:
            print("  - Extra in VAL:", extra_in_val)
        if extra_in_test:
            print("  - Extra in TEST:", extra_in_test)

    return device, dataset_dir


# Inizializzazione per esecuzione in modalità notebook: una sola chiamata a run_pipeline
# Per usare tutti i valori di default: cfg = Config()
# Per modificare solo alcuni parametri, passare solo quelli da sovrascrivere, es.:
#   cfg = Config(dataset_dir="altro_dataset")
#   cfg = Config(dataset_dir="dati/train_val_test", batch_size=32, num_workers=2)
cfg = Config()
device, dataset_dir = run_pipeline(cfg)

### Dataset e DataLoader

Si creano i dataset (ImageFolder) per train, val e test usando le trasformazioni baseline; per il training si usa anche una versione con augmentation (**aug_strong** in Exp03). Si costruiscono i DataLoader e si ricavano `class_names` e `num_classes` per il resto del notebook.

In [ ]:
# Trasformazioni: baseline per val/test, aug_strong per training (Exp03: augmentation piu' aggressiva)
t_baseline = Transforms(build_transforms(cfg.image_size, "baseline"))
t_aug = Transforms(build_transforms(cfg.image_size, "aug_strong"))

# Dataset: ImageFolder si aspetta train/val/test con sottocartelle per classe
trainset_base = ImageFolder(root=str(dataset_dir / "train"), transform=t_baseline)
valset = ImageFolder(root=str(dataset_dir / "val"), transform=t_baseline)
testset = ImageFolder(root=str(dataset_dir / "test"), transform=t_baseline)
trainset_aug = ImageFolder(root=str(dataset_dir / "train"), transform=t_aug)

# Nomi classi e numero (ordine alfabetico da ImageFolder)
class_names = trainset_base.classes
num_classes = len(class_names)
print("Num classes:", num_classes)
print("Train/Val/Test:", len(trainset_base), len(valset), len(testset))

# DataLoader: shuffle=True solo per training, num_workers da Config (0 in notebook evita problemi)
trainloader_base = DataLoader(trainset_base, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
trainloader_aug = DataLoader(trainset_aug, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
valloader = DataLoader(valset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
testloader = DataLoader(testset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers) 

### Visualizzazione campioni e augmentation

Prima di avviare il training e' utile ispezionare visivamente il dataset per verificare:
- che le immagini vengano caricate e normalizzate correttamente;
- l'effetto delle trasformazioni di augmentation rispetto all'originale.

**`back_to_image`** esegue la **denormalizzazione inversa** (applica media e std ImageNet al contrario) per riportare il tensore normalizzato a un array RGB visualizzabile con matplotlib.

**`show_samples`** mostra una griglia di campioni dal dataset con l'etichetta della classe sotto ogni immagine.

**`show_augmentation_comparison`** affianca l'immagine originale (baseline, senza augmentation) con un numero configurabile di versioni aumentate dello stesso campione, rendendo visivamente evidente l'effetto delle trasformazioni stocastiche (flip, crop, color jitter, ecc.).

In [ ]:
def back_to_image(img_tensor: torch.Tensor) -> np.ndarray:
    """
    Converte un tensore PyTorch normalizzato con i valori ImageNet
    in un array NumPy HxWx3 visualizzabile con matplotlib.

    Parametri
    ----------
    img_tensor : tensore di forma (3, H, W) gia' normalizzato

    Restituisce
    -----------
    Array float32 di forma (H, W, 3) con valori in [0, 1]
    """
    # Clona il tensore per non modificare l'originale
    img = img_tensor.clone().cpu()
    # Ricostruisce media e std nella forma corretta per broadcast (3, 1, 1)
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    # Denormalizzazione: x_orig = x_norm * std + mean
    img = (img * std + mean).clamp(0, 1)
    # Traspone da (C, H, W) a (H, W, C) come richiesto da matplotlib
    return img.numpy().transpose(1, 2, 0)


def show_samples(
    dataset,
    class_names: List[str],
    n_rows: int = 2,
    n_cols: int = 5,
    title: str = "Campioni",
    offset: int = 0,
) -> None:
    """
    Visualizza una griglia n_rows x n_cols di campioni estratti dal dataset.

    Parametri
    ----------
    dataset     : dataset PyTorch (es. ImageFolder con Transforms)
    class_names : lista dei nomi delle classi
    n_rows      : numero di righe della griglia
    n_cols      : numero di colonne della griglia
    title       : titolo della figura
    offset      : indice del primo campione da mostrare (utile per scorrere il dataset)
    """
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.2 * n_rows))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    for i in range(n_rows * n_cols):
        # Carica il campione all'indice (offset + i)
        img, label = dataset[offset + i]
        ax = axes[i // n_cols, i % n_cols]
        # Denormalizza e mostra l'immagine
        ax.imshow(back_to_image(img))
        ax.set_title(class_names[label], fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def show_augmentation_comparison(
    trainset_orig,
    trainset_aug,
    class_names: List[str],
    idx: int = 0,
    n_aug: int = 5,
) -> None:
    """
    Confronto visivo tra il campione originale (baseline) e n_aug versioni aumentate
    dello stesso indice nel dataset augmentato.

    Parametri
    ----------
    trainset_orig : dataset con trasformazioni baseline (senza augmentation)
    trainset_aug  : stesso dataset con trasformazioni aug_light (con augmentation)
    class_names   : lista dei nomi delle classi
    idx           : indice del campione da mostrare
    n_aug         : numero di versioni aumentate da affiancare all'originale
    """
    # Recupera l'immagine originale (senza augmentation)
    orig_img, label = trainset_orig[idx]
    class_name = class_names[label]

    fig, axes = plt.subplots(1, n_aug + 1, figsize=(3 * (n_aug + 1), 3.5))
    fig.suptitle(f"Augmentation — classe: {class_name}", fontsize=11, fontweight="bold")

    # Prima colonna: immagine originale in verde
    axes[0].imshow(back_to_image(orig_img))
    axes[0].set_title("Originale", color="green", fontweight="bold", fontsize=9)
    axes[0].axis("off")

    # Colonne successive: versioni aumentate (ogni chiamata a dataset[idx] produce aug diverse)
    for i in range(1, n_aug + 1):
        aug_img, _ = trainset_aug[idx]
        axes[i].imshow(back_to_image(aug_img))
        axes[i].set_title(f"Aug #{i}", fontsize=9)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


# Mostra una griglia di campioni dal training set (baseline, senza augmentation)
# per verificare che il caricamento e la normalizzazione siano corretti
show_samples(trainset_base, class_names, n_rows=2, n_cols=5,
             title="Training set — campioni baseline")

# Confronta l'immagine originale con le versioni aumentate
# (ogni esecuzione produce risultati diversi per le trasformazioni stocastiche)
show_augmentation_comparison(trainset_base, trainset_aug, class_names, idx=10, n_aug=5)


### Baseline CNN (benchmark)

Si addestra la **BaselineCNN** sul training set senza augmentation, con early stopping e ReduceLROnPlateau. Si carica il miglior checkpoint e si riportano accuracy su validation e test. Serve come riferimento per confrontare i modelli con transfer learning.

In [ ]:
# Baseline CNN: training senza augmentation, poi valutazione su val e test
print("\n=== BASELINE CNN (no augmentation) ===")
baseline = BaselineCNN(num_classes=num_classes).to(device)

# Crea la sottocartella dedicata alla baseline (checkpoints/baseline/)
baseline_ckpt_dir = Path(cfg.checkpoint_dir) / "baseline"
baseline_ckpt_dir.mkdir(parents=True, exist_ok=True)
baseline_ckpt_path = baseline_ckpt_dir / "best_baseline.pt"

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # smoothing riduce overconfidence e curve più morbide
optimizer = torch.optim.Adam(baseline.parameters(), lr=cfg.lr_head, weight_decay=cfg.weight_decay)
early = EarlyStopping(save_path=str(baseline_ckpt_path), patience=cfg.patience, min_delta=0.0)
history_base = train(
    baseline, trainloader_base, valloader,
    criterion, optimizer, device,
    epochs=cfg.epochs_head,
    early_stopping=early,
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs_head),
    scheduler_on="epoch"
)

# Carica il miglior checkpoint salvato da EarlyStopping e valuta
baseline.load_state_dict(torch.load(baseline_ckpt_path, map_location=device, weights_only=True))
val_loss, val_acc = test_epoch(baseline, valloader, criterion, device)
test_loss, test_acc = test_epoch(baseline, testloader, criterion, device)
print(f"Baseline -> VAL acc={val_acc:.4f} | TEST acc={test_acc:.4f}")



### Lista dei modelli da confrontare

Qui si definisce l'elenco dei modelli di transfer learning da addestrare e confrontare, ordinati in modo **cronologico** per evidenziare il progressivo miglioramento ottenibile con architetture piu' moderne:

| Modello | Anno | Tipo | Note |
|---|---|---|---|
| `vgg16` | 2014 | CNN classica | Punto di partenza storico |
| `resnet50` | 2015 | CNN con skip connections | Introduce i residual block |
| `efficientnet_b0` | 2019 | CNN con scaling uniforme | Compatto ed efficiente |
| `efficientnet_b4` | 2019 | CNN con scaling uniforme | Upgrade diretto di B0, piu' preciso |
| `convnext_tiny` | 2022 | CNN modernizzata | Ispirata ai Transformer, SOTA su ImageNet |
| `swin_tiny` | 2021 | Swin Transformer | Attention con finestre locali scorrevoli |

La lista e' modificabile: per test veloci si possono lasciare solo 2-3 modelli (es. `["resnet50", "convnext_tiny"]`). I checkpoint verranno salvati in `checkpoints/{nome_modello}/` (best_head.pt e best_finetune.pt).

In [ ]:
# Elenco dei modelli candidati per il transfer learning (Exp03: tutti e 6),
# ordinati cronologicamente per mostrare la progressione storica delle architetture.
MODEL_CANDIDATES = [
    "vgg16",            # CNN classica (2014) - punto di partenza storico
    "resnet50",         # CNN con skip connections (2015)
    "efficientnet_b0",  # Scaling uniforme compatto (2019)
    "efficientnet_b4",  # Scaling uniforme piu' grande (2019) - upgrade diretto di B0
    "convnext_tiny",    # CNN modernizzata ispirata ai Transformer (2022)
    "swin_tiny",        # Swin Transformer con finestre locali (2021)
]

### Loop: addestramento di tutti i modelli candidati (fault-tolerant)

Per ogni modello in `MODEL_CANDIDATES` si eseguono due fasi:
1. **Head-only**: il backbone è congelato, si addestra solo il classificatore finale (salva `best_head.pt`).
2. **Fine-tuning**: gli ultimi blocchi del backbone vengono sbloccati con `unfreeze_last_blocks` e si riaddestra con un learning rate molto più basso (salva `best_finetune.pt`).

**Novità v1 — Progressive Checkpointing (fault tolerance):**

Ogni fase di training salva automaticamente un checkpoint periodico (ogni `cfg.checkpoint_every` epoch, default = 5) nella stessa cartella del best model, con il nome `resume_head.pt` o `resume_finetune.pt`. Se il training viene interrotto (crash, kernel stop, timeout Colab), basta rieseguire la stessa cella: la funzione `train()` rileverà il checkpoint e riprenderà dall'epoch successiva a quella salvata, senza perdere il lavoro già svolto.

Al termine del loop, `results` raccoglie metriche (val_acc, test_acc) e storie di training per tutti i modelli.

> **Nota sui tempi**: l'esecuzione può richiedere diversi minuti per ogni modello; su GPU (Google Colab) è significativamente più veloce che su CPU.

> **Nota tecnica — Compatibilità Apple Silicon (MPS):**
> PyTorch su Apple Silicon (chip M1/M2/M3) utilizza il backend MPS per l'accelerazione GPU.
> Alcune architetture moderne come **ConvNeXt** e **Swin Transformer** presentano un bug noto
> nel backward pass su MPS: internamente usano `view()` su tensori non contigui in memoria,
> causando un `RuntimeError`. Il problema è documentato su GitHub PyTorch (issue #96056).
> **Soluzione adottata**: quando il device è MPS e il modello è `convnext_tiny` o `swin_tiny`,
> la fase di fine-tuning viene eseguita automaticamente su CPU. La valutazione finale
> viene comunque eseguita su MPS. Su GPU NVIDIA (Google Colab) o CPU il problema non si presenta.

In [ ]:
# Lista che conterrà per ogni modello: nome, val_acc, test_acc, history_head, history_finetune
results = []

for model_name in MODEL_CANDIDATES:
    # Crea sottocartella checkpoints/{model_name}/ per best_head.pt e best_finetune.pt
    ckpt_dir = Path(cfg.checkpoint_dir) / model_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # Fase 1: solo head trainabile (backbone congelato), ottimizzatore solo sui parametri con requires_grad
    tl = build_transfer_model(model_name, num_classes=num_classes, device=device, freeze_backbone=True)
    criterion_tl = nn.CrossEntropyLoss(label_smoothing=0.1)  # smoothing riduce overconfidence e curve più morbide
    optimizer_tl = torch.optim.Adam(
        filter(lambda p: p.requires_grad, tl.parameters()),
        lr=cfg.lr_head,
        weight_decay=cfg.weight_decay,
    )
    path_head = str(ckpt_dir / "best_head.pt")
    early_tl = EarlyStopping(save_path=path_head, patience=cfg.patience, min_delta=0.0)

    print(f"\n=== TRANSFER LEARNING ({model_name}) + DATA AUGMENTATION ===")
    # Percorso checkpoint periodico per la fase head (fault tolerance)
    resume_head_path = str(ckpt_dir / "resume_head.pt")
    history_head = train(
        tl, trainloader_aug, valloader,
        criterion_tl, optimizer_tl, device,
        epochs=cfg.epochs_head,
        early_stopping=early_tl,
        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_tl, T_max=cfg.epochs_head),
        scheduler_on="epoch",
        resume_ckpt_path=resume_head_path,     # riprende da qui se il training era interrotto
        checkpoint_every=cfg.checkpoint_every, # salva checkpoint periodico ogni N epoche
    )
    # Ricarica miglior checkpoint della fase head e valuta su validation
    tl.load_state_dict(torch.load(path_head, map_location=device, weights_only=True))
    _, val_acc_head = test_epoch(tl, valloader, criterion_tl, device)
    print(f"TL (head only) -> VAL acc={val_acc_head:.4f}")

    # Fase 2: fine-tuning: sblocca ultimi blocchi del backbone, learning rate più basso
    unfreeze_last_blocks(tl, model_name, num_blocks=1)

    # Workaround MPS: ConvNeXt e Swin Transformer hanno un bug nel backward pass su
    # Apple Silicon (view su tensori non contigui). Per questi modelli il fine-tuning
    # viene eseguito su CPU; head-only e valutazione finale restano su MPS.
    MPS_PROBLEMATIC = ('convnext_tiny', 'swin_tiny')
    ft_device = torch.device('cpu') if (device.type == 'mps' and model_name in MPS_PROBLEMATIC) else device
    if ft_device != device:
        print(f"  [MPS workaround] Fine-tuning di {model_name} su CPU (bug PyTorch MPS backward)")
        tl = tl.to(ft_device)

    optimizer_ft = torch.optim.Adam(
        filter(lambda p: p.requires_grad, tl.parameters()),
        lr=cfg.lr_finetune,
        weight_decay=cfg.weight_decay,
    )
    path_finetune = str(ckpt_dir / "best_finetune.pt")
    early_ft = EarlyStopping(save_path=path_finetune, patience=cfg.patience, min_delta=0.0)

    print(f"\n=== FINE-TUNING ({model_name}) ===")
    # Percorso checkpoint periodico per la fase fine-tuning (fault tolerance)
    resume_ft_path = str(ckpt_dir / "resume_finetune.pt")
    history_finetune = train(
        tl, trainloader_aug, valloader,
        criterion_tl, optimizer_ft, ft_device,
        epochs=cfg.epochs_finetune,
        early_stopping=early_ft,
        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=cfg.epochs_finetune),
        scheduler_on="epoch",
        resume_ckpt_path=resume_ft_path,       # riprende da qui se il training era interrotto
        checkpoint_every=cfg.checkpoint_every, # salva checkpoint periodico ogni N epoche
    )
    # Ricarica miglior checkpoint dopo fine-tuning e riporta il modello su device principale
    tl.load_state_dict(torch.load(path_finetune, map_location=ft_device, weights_only=True))
    tl = tl.to(device)  # Riporta sempre su device principale per la valutazione
    val_loss_f, val_acc_f = test_epoch(tl, valloader, criterion_tl, device)
    test_loss_f, test_acc_f = test_epoch(tl, testloader, criterion_tl, device)
    print(f"[{model_name}] VAL acc={val_acc_f:.4f} | TEST acc={test_acc_f:.4f}")

    # Salva in results per la tabella e per i grafici delle curve
    results.append({
        "model_name": model_name,
        "val_acc": val_acc_f,
        "test_acc": test_acc_f,
        "history_head": history_head,
        "history_finetune": history_finetune,
    })

### Riepilogo e scelta del migliore

Si visualizza una tabella con le metriche (val_acc, test_acc) di ogni modello e si sceglie il **migliore in base alla validation accuracy** (senza usare il test per la scelta). Il nome del modello selezionato viene salvato in `best_model_name` per le celle successive.

In [ ]:
# Tabella di confronto: val_acc e test_acc per ogni modello
df_results = pd.DataFrame([{"modello": r["model_name"], "val_acc": r["val_acc"], "test_acc": r["test_acc"]} for r in results])
print("Confronto modelli (dopo fine-tuning):")
print(df_results.to_string(index=False))

# Scelta del migliore in base alla validation accuracy (il test non viene usato per la scelta)
best_record = max(results, key=lambda x: x["val_acc"])
best_model_name = best_record["model_name"]
print(f"\nMiglior modello (per val_acc): {best_model_name} | val_acc={best_record['val_acc']:.4f} | test_acc={best_record['test_acc']:.4f}")

### Curve di training del modello migliore

Si mostrano le curve di train/val loss e val accuracy per il modello selezionato (`best_model_name`), nelle due fasi: **Head only** (solo testa addestrata) e **Fine-tuning** (backbone parzialmente sbloccato). Serve a verificare visivamente se c’è overfitting o underfitting.

In [ ]:
# Recupera dalla lista results le history del modello scelto (best_model_name)
best_record = next(r for r in results if r["model_name"] == best_model_name)
# Grafici della fase "solo head" (backbone congelato)
plot_training_curves(best_record["history_head"], title=f"{best_model_name} - Head only")
# Grafici della fase fine-tuning (backbone parzialmente sbloccato)
plot_training_curves(best_record["history_finetune"], title=f"{best_model_name} - Fine-tuning")

### Valutazione finale sul test set (modello migliore)

Si carica il **modello selezionato** (pesi da `checkpoints/{best_model_name}/best_finetune.pt`), si eseguono le predizioni sul test set e si producono la **matrice di confusione** e il **classification report**. Il test set viene usato solo per questa valutazione finale del modello scelto in base alla validation.

In [ ]:
# Ricostruisci il modello migliore (stessa architettura) e carica i pesi da checkpoints/{best_model_name}/
best_tl = build_transfer_model(best_model_name, num_classes=num_classes, device=device, freeze_backbone=False)
path_best = Path(cfg.checkpoint_dir) / best_model_name / "best_finetune.pt"
best_tl.load_state_dict(torch.load(str(path_best), map_location=device, weights_only=True))

# Predizioni su tutto il test set, poi matrice di confusione e report (precision, recall, f1)
y_true, y_pred = predict_all(best_tl, testloader, device)
plot_confusion_matrix(y_true, y_pred, class_names, normalize=False)

print("\nClassification report (TEST) - modello:", best_model_name)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

### Generazione automatica del report HTML

Questa cella genera automaticamente un **report HTML completo** ogni volta che il notebook viene eseguito. Il file viene salvato nella cartella `reports/` con il nome `report_YYYY-MM-DD_HH-MM-SS.html`.

Il report è **auto-contenuto**: tutte le immagini (curve di training, matrice di confusione) sono incorporate come base64, quindi il file si apre in qualsiasi browser senza bisogno di file aggiuntivi.

**Contenuto del report:**
- Data e ora di generazione
- Parametri di configurazione usati (`Config`)
- Dimensioni del dataset e lista delle classi
- Risultati della baseline CNN (val_acc, test_acc)
- Tabella di confronto tra tutti i modelli TL (con il migliore evidenziato)
- Curve di training del modello migliore (fase head e fine-tuning)
- Matrice di confusione sul test set
- Classification report completo (precision, recall, F1 per classe)

Per aprire il report: aprire il file `.html` nella cartella `reports/` con un browser (doppio clic o drag-and-drop).

In [ ]:
def generate_report() -> None:
    """
    Genera un report HTML completo con tutti i risultati del notebook e lo salva
    nella cartella reports/ con nome timestampato (report_YYYY-MM-DD_HH-MM-SS.html).
    Il report e' auto-contenuto: le immagini sono incluse come base64 e il file
    si apre in qualsiasi browser senza file aggiuntivi. Usa le variabili globali
    gia' disponibili al termine dell'esecuzione del notebook (cfg, results,
    best_model_name, best_record, y_true, y_pred, class_names, val_acc, test_acc).
    """

    # --- 1. Preparazione cartella reports/ e nome file con timestamp ---
    reports_dir = Path("reports")
    reports_dir.mkdir(exist_ok=True)
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    report_path = reports_dir / f"report_{timestamp}.html"

    # --- 2. Helper: converte una figura matplotlib in stringa base64 PNG ---
    def fig_to_base64(fig: plt.Figure) -> str:
        """Salva la figura in un buffer in memoria e restituisce la stringa base64."""
        buf = io.BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight")
        buf.seek(0)
        return base64.b64encode(buf.read()).decode("utf-8")

    # --- 3. Figura: curve di training del modello migliore (fase head) ---
    fig_head, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    h = best_record["history_head"]
    eps = range(1, len(h["train_loss"]) + 1)
    ax1.plot(eps, h["train_loss"], label="Train loss", color="C0")
    ax1.plot(eps, h["val_loss"], label="Val loss", color="C1")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()
    ax1.set_title(f"{best_model_name} - Head only (Loss)")
    ax2.plot(eps, h["val_acc"], label="Val accuracy", color="C2")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend()
    ax2.set_title(f"{best_model_name} - Head only (Accuracy)")
    plt.tight_layout()
    img_head = fig_to_base64(fig_head)
    plt.close(fig_head)

    # --- 4. Figura: curve di training del modello migliore (fase fine-tuning) ---
    fig_ft, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    h = best_record["history_finetune"]
    eps = range(1, len(h["train_loss"]) + 1)
    ax1.plot(eps, h["train_loss"], label="Train loss", color="C0")
    ax1.plot(eps, h["val_loss"], label="Val loss", color="C1")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()
    ax1.set_title(f"{best_model_name} - Fine-tuning (Loss)")
    ax2.plot(eps, h["val_acc"], label="Val accuracy", color="C2")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend()
    ax2.set_title(f"{best_model_name} - Fine-tuning (Accuracy)")
    plt.tight_layout()
    img_ft = fig_to_base64(fig_ft)
    plt.close(fig_ft)

    # --- 5. Figura: matrice di confusione con conteggi assoluti ---
    cm_raw = confusion_matrix(y_true, y_pred)
    n = len(class_names)
    fig_cm, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_raw, cmap="Blues")
    plt.colorbar(im, ax=ax)
    thresh = cm_raw.max() / 2.0
    for i in range(n):
        for j in range(n):
            val = int(cm_raw[i, j])
            ax.text(j, i, str(val), ha="center", va="center", fontsize=8,
                    color="white" if val > thresh else "black")
    ax.set_xticks(range(n)); ax.set_xticklabels(class_names, rotation=90, fontsize=10)
    ax.set_yticks(range(n)); ax.set_yticklabels(class_names, fontsize=10)
    ax.set_xlabel("Predicted label", fontsize=12)
    ax.set_ylabel("True label", fontsize=12)
    ax.set_title(f"Confusion Matrix - {best_model_name}", fontsize=14)
    plt.tight_layout()
    img_cm = fig_to_base64(fig_cm)
    plt.close(fig_cm)

    # --- 6. Righe HTML per la tabella confronto modelli (best evidenziato in verde) ---
    rows_html = ""
    for r in results:
        highlight = "background:#d4edda;" if r["model_name"] == best_model_name else ""
        rows_html += (
            f"<tr style='{highlight}'>"
            f"<td>{r['model_name']}</td>"
            f"<td>{r['val_acc']:.4f}</td>"
            f"<td>{r['test_acc']:.4f}</td>"
            "</tr>"
        )

    # --- 7. Classification report come testo preformattato ---
    cr_text = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    cr_html = "<pre>" + cr_text + "</pre>"

    # --- 8. Righe HTML per la tabella parametri Config ---
    cfg_html = "".join(
        f"<tr><td>{k}</td><td>{v}</td></tr>"
        for k, v in vars(cfg).items()
    )

    # --- 9. Assemblaggio del documento HTML completo ---
    html = f"""<!DOCTYPE html>
<html lang="it">
<head>
  <meta charset="utf-8">
  <title>GourmetAI Report {timestamp}</title>
  <style>
    body {{ font-family: sans-serif; margin: 40px; color: #222; }}
    h1 {{ color: #2c3e50; border-bottom: 2px solid #2c3e50; padding-bottom: 8px; }}
    h2 {{ color: #34495e; margin-top: 40px; border-bottom: 1px solid #ccc; padding-bottom: 4px; }}
    table {{ border-collapse: collapse; width: 100%; margin-bottom: 20px; }}
    th, td {{ border: 1px solid #ccc; padding: 8px 12px; text-align: left; }}
    th {{ background: #f0f0f0; font-weight: bold; }}
    tr:nth-child(even) {{ background: #fafafa; }}
    img {{ max-width: 100%; border: 1px solid #ddd; border-radius: 4px; margin: 8px 0; }}
    pre {{ background: #f8f8f8; padding: 16px; border-radius: 4px; overflow-x: auto; font-size: 13px; }}
    .badge {{ display:inline-block; background:#27ae60; color:#fff; padding:2px 10px; border-radius:12px; font-size:13px; }}
  </style>
</head>
<body>

<h1>GourmetAI &ndash; Report di classificazione immagini di cibo</h1>
<p><b>Generato il:</b> {timestamp}</p>

<h2>1. Configurazione</h2>
<table>
  <tr><th>Parametro</th><th>Valore</th></tr>
  {cfg_html}
</table>

<h2>2. Dataset</h2>
<p>
  <b>Numero di classi:</b> {num_classes} &nbsp;|&nbsp;
  <b>Train:</b> {len(trainset_base)} immagini &nbsp;|&nbsp;
  <b>Val:</b> {len(valset)} immagini &nbsp;|&nbsp;
  <b>Test:</b> {len(testset)} immagini
</p>
<p><b>Classi:</b> {", ".join(class_names)}</p>

<h2>3. Baseline CNN (benchmark)</h2>
<p>VAL accuracy = <b>{val_acc:.4f}</b> &nbsp;|&nbsp; TEST accuracy = <b>{test_acc:.4f}</b></p>

<h2>4. Confronto modelli Transfer Learning</h2>
<table>
  <tr><th>Modello</th><th>Val Acc</th><th>Test Acc</th></tr>
  {rows_html}
</table>
<p>
  <b>Miglior modello selezionato:</b>
  <span class="badge">{best_model_name}</span>
  &nbsp; val_acc = <b>{best_record['val_acc']:.4f}</b>
  &nbsp;|&nbsp; test_acc = <b>{best_record['test_acc']:.4f}</b>
</p>

<h2>5. Curve di training &ndash; {best_model_name} (Head only)</h2>
<img src="data:image/png;base64,{img_head}" alt="Curve training head"/>

<h2>6. Curve di training &ndash; {best_model_name} (Fine-tuning)</h2>
<img src="data:image/png;base64,{img_ft}" alt="Curve training finetune"/>

<h2>7. Confusion Matrix &ndash; {best_model_name} (Test set)</h2>
<img src="data:image/png;base64,{img_cm}" alt="Confusion matrix"/>

<h2>8. Classification Report &ndash; {best_model_name} (Test set)</h2>
{cr_html}

</body>
</html>"""

    # Scrittura del file HTML su disco
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Report generato: {report_path}")
    print("Aprilo con un browser (doppio clic sul file .html).")


# Esecuzione automatica al termine del notebook
generate_report()

---

# Suggerimenti per migliorare la precisione

Di seguito alcune **possibili migliorie** da esplorare per aumentare la precisione del classificatore:

1. **Data augmentation più ricca**  
   Provare la modalità `aug_strong` al posto di `aug_light` per il training, o introdurre mixup/cutmix. Attenzione a non esagerare per non degradare la qualità delle feature.

2. **Più epoch e patience**  
   Aumentare `epochs_head` e `epochs_finetune` (e eventualmente `patience`) per dare al modello più tempo di convergenza, soprattutto se le curve di training non si sono ancora stabilizzate.

3. **Learning rate e scheduler**  
   Sperimentare learning rate diversi (es. 5e-4 per la head, 1e-4 per il fine-tuning) e scheduler (CosineAnnealingLR, OneCycleLR) al posto di ReduceLROnPlateau.

4. **Unfreeze progressivo**  
   Sbloccare più blocchi nel fine-tuning (`num_blocks` maggiore per VGG/EfficientNet) o sbloccare layer per layer con learning rate differenziati (più bassi per i layer iniziali).

5. **Modelli più grandi o ensemble**  
   Aggiungere candidati come ResNet101, EfficientNet-B2/B3, o fare ensemble (media dei logit o voto a maggioranza) tra i migliori modelli.

6. **Bilanciamento delle classi**  
   Se alcune classi sono sotto-rappresentate, usare pesi per la loss (`weight` in CrossEntropyLoss) o campionamento bilanciato (WeightedRandomSampler).

7. **Risoluzione e input size**  
   Aumentare `image_size` (es. 384 o 448) se la GPU lo consente, per sfruttare meglio i backbone preaddestrati (che spesso vedono immagini a 224 o più).

8. **Test Time Augmentation (TTA)**  
   In fase di test, applicare più trasformazioni (flip, piccole rotazioni) e mediare le predizioni per aumentare la robustezza.

9. **Regolarizzazione**  
   Aumentare leggermente `weight_decay` o usare dropout più alto nella head per ridurre l’overfitting se la validation loss sale mentre la train loss scende.

10. **Dati e pulizia**  
    Verificare la qualità del dataset (etichette errate, immagini fuori tema) e considerare l’aggiunta di nuovi dati o l’uso di un dataset più grande per il pre-training o il fine-tuning.